# Quickstart with Ray AI Runtime

<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Generic/ray_logo.png" width="20%" loading="lazy">

## Preliminaries

### Install libraries

In [1]:
#! pip install -U ray==2.3.0 xgboost_ray==0.1.18

### Imports

In [2]:
import ray
from ray.air.config import ScalingConfig
from ray.data.preprocessors import MinMaxScaler
from ray.train.xgboost import XGBoostTrainer

### Initialize Ray runtime

In [ ]:
ray.init()

2026-04-20 20:50:09,743	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8266 
/home/sudarsun/venv/mlops/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.3
Ray version:,2.54.1
Dashboard:,http://127.0.0.1:8266


(TrainController pid=130555) Attempting to start training worker group of size 8 with the following resources: [{'CPU': 1}] * 8
(TrainController pid=130555) Started training worker group of size 8: 
(TrainController pid=130555) - (ip=192.168.1.37, pid=130689) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=130555) - (ip=192.168.1.37, pid=130690) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=130555) - (ip=192.168.1.37, pid=130688) world_rank=2, local_rank=2, node_rank=0
(TrainController pid=130555) - (ip=192.168.1.37, pid=130679) world_rank=3, local_rank=3, node_rank=0
(TrainController pid=130555) - (ip=192.168.1.37, pid=130677) world_rank=4, local_rank=4, node_rank=0
(TrainController pid=130555) - (ip=192.168.1.37, pid=130678) world_rank=5, local_rank=5, node_rank=0
(TrainController pid=130555) - (ip=192.168.1.37, pid=130684) world_rank=6, local_rank=6, node_rank=0
(TrainController pid=130555) - (ip=192.168.1.37, pid=130680) world_rank=7, local_rank=7, node_

## Load and prepare data with Ray Datasets

### Read Parquet file to Ray Dataset

In [4]:
dataset = ray.data.read_parquet(
    "s3://anyscale-training-data/intro-to-ray-air/nyc_taxi_2021.parquet"
)

Parquet dataset sampling:   0%|          | 0.00/1.00 [00:00<?, ? file/s]

2026-04-20 20:51:10,519	INFO parquet_datasource.py:1079 -- Estimated parquet encoding ratio is 6.232.
2026-04-20 20:51:10,520	INFO parquet_datasource.py:1139 -- Estimated parquet reader batch size at 2354697 rows


In [5]:
dataset

shape: (?, 8)
╭─────────────────┬───────────────┬─────────────┬───────────────┬───────┬─────────────┬────────────┬───────────────────╮
│ passenger_count ┆ trip_distance ┆ fare_amount ┆ trip_duration ┆ hour  ┆ day_of_week ┆ is_big_tip ┆ __index_level_0__ │
│ ---             ┆ ---           ┆ ---         ┆ ---           ┆ ---   ┆ ---         ┆ ---        ┆ ---               │
│ double          ┆ double        ┆ double      ┆ int64         ┆ int64 ┆ int64       ┆ bool       ┆ int64             │
╰─────────────────┴───────────────┴─────────────┴───────────────┴───────┴─────────────┴────────────┴───────────────────╯
(Dataset isn't materialized)

Returned `dataset` is [Ray Dataset](https://docs.ray.io/en/latest/data/api/doc/ray.data.Dataset.html#ray-data-dataset) - standard way to load and exchange data in Ray AI Runtime.

In AIR, Datasets are used extensively for data loading and transformation. They are meant as a last-mile bridge from ETL pipeline outputs to distributed applications and libraries in Ray.

### Split data into training and validation subsets

In [6]:
train_dataset, valid_dataset = dataset.train_test_split(test_size=0.3)

2026-04-20 20:51:11,204	INFO logging.py:392 -- Registered dataset logger for dataset dataset_1_0
2026-04-20 20:51:11,221	INFO streaming_executor.py:182 -- Starting execution of Dataset dataset_1_0. Full logs are in /tmp/ray/session_2026-04-20_20-50-04_121056_126032/logs/ray-data
2026-04-20 20:51:11,222	INFO streaming_executor.py:183 -- Execution plan of Dataset dataset_1_0: InputDataBuffer[Input] -> TaskPoolMapOperator[ReadParquet] -> AggregateNumRows[AggregateNumRows]
2026-04-20 20:51:11,225	WARNING resource_manager.py:141 -- ⚠️  Ray's object store is configured to use only 42.9% of available memory (14.0GiB out of 32.7GiB total). For optimal Ray Data performance, we recommend setting the object store to at least 50% of available memory. You can do this by setting the 'object_store_memory' parameter when calling ray.init() or by setting the RAY_DEFAULT_OBJECT_STORE_MEMORY_PROPORTION environment variable.
2026-04-20 20:51:11,227	INFO __init__.py:56 -- Progress will be logged because st

### Split datasets into blocks for parallel preprocessing

In [7]:
train_dataset = train_dataset.repartition(num_blocks=3)
valid_dataset = valid_dataset.repartition(num_blocks=3)

`num_blocks` should be lower than number of cores in the cluster

### Define a preprocessor to normalize the columns by their range

In [8]:
preprocessor = MinMaxScaler(columns=["trip_distance", "trip_duration"])

[Preprocessors](https://docs.ray.io/en/latest/ray-air/key-concepts.html#preprocessors) are primitives that transform input data into features. They operate on Datasets, making them scalable and compatible with a variety of datasources and dataframe libraries.

Ray AI Runtime comes with a collection of built-in preprocessors, and you can also define your own with simple templates (see [Using preprocessors](https://docs.ray.io/en/latest/ray-air/preprocessors.html) for more information).

## Train the model with Ray Train

|<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Scaling_model_training/data_parallelism.png" width="50%" loading="lazy">|
|:--|
|Ray Train provides distributed data parallel training capabilities. A large dataset is sharded across multiple worker nodes each containing a model copy. Gradients calculated on independent nodes are continuously synchronized with others to produce a final trained model.|

### Create XGBoost trainer

In [9]:
import xgboost

import ray.data
import ray.train
from ray.train.xgboost import RayTrainReportCallback
from ray.train.xgboost import XGBoostTrainer

def train_fn_per_worker(config: dict):
    # (Optional) Add logic to resume training state from a checkpoint.
    # ray.train.get_checkpoint()

    # 1. Get the dataset shard for the worker and convert to a `xgboost.DMatrix`
    train_ds_iter, eval_ds_iter = (
        ray.train.get_dataset_shard("train"),
        ray.train.get_dataset_shard("validation"),
    )
    train_ds, eval_ds = train_ds_iter.materialize(), eval_ds_iter.materialize()

    train_df, eval_df = train_ds.to_pandas(), eval_ds.to_pandas()
    train_X, train_y = train_df.drop(["is_big_tip"], axis=1), train_df["is_big_tip"]
    eval_X, eval_y = eval_df.drop(["is_big_tip"], axis=1), eval_df["is_big_tip"]

    dtrain = xgboost.DMatrix(train_X, label=train_y)
    deval = xgboost.DMatrix(eval_X, label=eval_y)

    params = {
        "eta": 1e-4,
        "subsample": 0.5,
        "max_depth": 3,
        "objective": "binary:logistic",
        "eval_metric": ["logloss", "error"],     
        "tree_method": "approx",  # use "gpu_hist" for GPU training
    }

    # 2. Do distributed data-parallel training.
    # Ray Train sets up the necessary coordinator processes and
    # environment variables for your workers to communicate with each other.
    bst = xgboost.train(
        params,
        dtrain=dtrain,
        evals=[(deval, "validation"), (dtrain, "train")],
        num_boost_round=20,
        callbacks=[RayTrainReportCallback()],
    )

In [10]:
trainer = XGBoostTrainer(
    train_fn_per_worker,
    datasets={"train": train_dataset, "validation": valid_dataset},
    scaling_config=ray.train.ScalingConfig(num_workers=8),
   
)

During training, `trainer` will use `num_blocks` workers, defined when repartitioning dataset.

Ray AI Runtime comes with built-in integrations with mang popular ML projects like PyTorch, Keras, LightGBM and more. Refer to the [Ray Train docs](https://docs.ray.io/en/latest/train/train.html#quick-start-to-distributed-training-with-ray-train) for more details. Optionally, read more about the Ray-XGBoost integration in the [Introducing Distributed XGBoost Training with Ray](https://www.anyscale.com/blog/distributed-xgboost-training-with-ray) blog post.

### Invoke training - this is computationally intensive operation

In [11]:
result = trainer.fit()

(pid=131261) Running Dataset train_6_0.: 0.00 row [00:00, ? row/s]

(pid=131261) - Repartition:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=131261)   *- Split Repartition:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=131261) - split(8, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=131260) Running Dataset validation_8_0.: 0.00 row [00:00, ? row/s]

...

(pid=130690) Running Dataset dataset_25_0.: 0.00 row [00:00, ? row/s]

(pid=130678) Running Dataset dataset_20_0.: 0.00 row [00:00, ? row/s]

(pid=130677) Running Dataset dataset_23_0.: 0.00 row [00:00, ? row/s]

(pid=130680) Running Dataset dataset_24_0.: 0.00 row [00:00, ? row/s]

(pid=130684) Running Dataset dataset_21_0.: 0.00 row [00:00, ? row/s]

The resulting object grants access to metrics, checkpoints, and errors

### Report results

In [12]:
print(f"train acc = {1 - result.metrics['train-error']:.4f}")
print(f"valid acc = {1 - result.metrics['validation-error']:.4f}")


train acc = 0.5941
valid acc = 0.5964


## Shutdown Ray runtime

In [13]:
ray.shutdown()

Disconnect the worker and terminate processes started by `ray.init()`.

# Connect with the Ray community

You can learn and get more involved with the Ray community of developers and researchers:

* [**Ray documentation**](https://docs.ray.io/en/latest)

* [**Official Ray site**](https://www.ray.io/)  
Browse the ecosystem and use this site as a hub to get the information that you need to get going and building with Ray.

* [**Join the community on Slack**](https://forms.gle/9TSdDYUgxYs8SA9e8)  
Find friends to discuss your new learnings in our Slack space.

* [**Use the discussion board**](https://discuss.ray.io/)  
Ask questions, follow topics, and view announcements on this community forum.

* [**Join a meetup group**](https://www.meetup.com/Bay-Area-Ray-Meetup/)  
Tune in on meet-ups to listen to compelling talks, get to know other users, and meet the team behind Ray.

* [**Open an issue**](https://github.com/ray-project/ray/issues/new/choose)  
Ray is constantly evolving to improve developer experience. Submit feature requests, bug-reports, and get help via GitHub issues.

* [**Become a Ray contributor**](https://docs.ray.io/en/latest/ray-contribute/getting-involved.html)  
We welcome community contributions to improve our documentation and Ray framework.

<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Generic/ray_logo.png" width="20%" loading="lazy">